### Replace the citation numbers in a saved Perplexity dialogue with matching literature note or zotero item links

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#savemychatbot_perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_single_prompt_savemychatbot_example.md'
savemychatbot_perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_multi_prompt_savemychatbot_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file_perplex = tmp_dir / "tmp_new_cites_perplexity_example.md"  # processed raw perplexity output
#output_file_savemychatbot = tmp_dir / "tmp_savemychatbot_perplexity_example.md"  # processed savemychatbot 
output_file_savemychatbot = tmp_dir / 'tmp_savemychatbot_multiprompt_perplexity_example.md'

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


In [3]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [4]:
# Make a lookup dict: zotero DB item URL to bibtex citekey
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [5]:

set([parent['data']['itemType'] for parent in parentItems])

{'blogPost',
 'book',
 'bookSection',
 'computerProgram',
 'conferencePaper',
 'dataset',
 'document',
 'email',
 'encyclopediaArticle',
 'forumPost',
 'journalArticle',
 'magazineArticle',
 'manuscript',
 'newspaperArticle',
 'note',
 'preprint',
 'presentation',
 'report',
 'thesis',
 'videoRecording',
 'webpage'}

In [6]:
# Collect info about each zotero DB item that has a URL


lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}

zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if not (title := pdat.get('title')):
        continue

    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], title=title, hasLitNote=citekeyThis in lit_note_file_stems))

    if url := pdat.get('url'):
        if normalized_url :=rfw.normalize_url(url):
            citekeysForURL[normalized_url].append(citekeyThis)

if repeatedURLs := {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}:
    print(f"Found {len(repeatedURLs)} URLs with > 1 parent (citekey)")
    for url, citekeys in repeatedURLs.items():
        print(f"{', '.join(citekeys)}\n\t{url}")
    raise Exception(f'Not built for repeated URLS')

citekey_to_url = {citekeys[0]: url for url, citekeys in citekeysForURL.items()}
url_to_citekey = {url: citekey for citekey, url in citekey_to_url.items()}

zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

if sum(hasNoURL := zot_db_items.url.isna()):
    print(f"Dropping {sum(hasNoURL)} of {len(zot_db_items)} zotero entries with no URL:")
    display((zot_db_items_no_url := zot_db_items[hasNoURL]).head())
    zot_db_items = zot_db_items[~hasNoURL]


Dropping 149 of 1681 zotero entries with no URL:


,citekey,zotkey,title,hasLitNote,url
0,MMSDataModelSummary_v5.2,8XJHRYMU,MMS Data Model Package Summary v5.2,False,NaN
15,Seals99irradFrcstDiag,WYP9J7EU,The heart of suny irradiance forecasting,False,NaN
92,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,Testing increases suggestibility for narrative...,False,NaN
153,Gaur20attribModellingRvw,HLHKVCLX,Attribution modelling in marketing: Literature...,False,NaN
200,Wang19predOptcoolLdFrcst,R7TJLE7Y,Cooling load forecasting-based predictive opti...,False,NaN


In [7]:
#zot_db_items.url.isna()
zot_db_items

,citekey,zotkey,title,hasLitNote,url
1,Gerber09normsMotiveVote,6V73VWME,Descriptive social norms and motivation to vot...,False,https://www.journals.uchicago.edu/doi/abs/10.1...
2,Poston13politAdsVizAuralMeaning,6VPD3STC,Political advertising in the 2012 presidential...,False,https://scholarworks.boisestate.edu/td/602
3,Deaton21altruisByCountry,YIN5FL7M,An exploration of global altruistic variations...,False,https://rave.ohiolink.edu/etdc/view?acc_num=xu...
4,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,Topic analysis in news via sparse learning: a ...,False,https://www.sciencedirect.com/science/article/...
5,Jarvis23gutenbergInternet,ETBJL3D9,The Gutenberg Parenthesis: The age of print an...,False,https://www.bloomsbury.com/us/gutenberg-parent...
...,...,...,...,...,...
1670,Oracle19evDetDisaggAMI,JI47UFCR,AMI-based EV Detection & Disaggregation,False,https://www.oracle.com/a/ocom/docs/industries/...
1672,Bidgely19amInsightsRprt,EKLZCYP5,AMI-Driven Insights Report,False,https://www.idcutilitiessummit.com/index/resou...
1674,Hare18disaggHmLdDmdResp,WA8IQAXP,Disaggregation of residential home energy via ...,False,https://dspace.mit.edu/handle/1721.1/117983
1675,Rehman21LoadDisaggThesis,8MAAZN8P,Load Disaggregation: Towards Energy Efficient ...,False,https://openrepository.aut.ac.nz/handle/10292/...


In [8]:
zot_db_items

,citekey,zotkey,title,hasLitNote,url
1,Gerber09normsMotiveVote,6V73VWME,Descriptive social norms and motivation to vot...,False,https://www.journals.uchicago.edu/doi/abs/10.1...
2,Poston13politAdsVizAuralMeaning,6VPD3STC,Political advertising in the 2012 presidential...,False,https://scholarworks.boisestate.edu/td/602
3,Deaton21altruisByCountry,YIN5FL7M,An exploration of global altruistic variations...,False,https://rave.ohiolink.edu/etdc/view?acc_num=xu...
4,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,Topic analysis in news via sparse learning: a ...,False,https://www.sciencedirect.com/science/article/...
5,Jarvis23gutenbergInternet,ETBJL3D9,The Gutenberg Parenthesis: The age of print an...,False,https://www.bloomsbury.com/us/gutenberg-parent...
...,...,...,...,...,...
1670,Oracle19evDetDisaggAMI,JI47UFCR,AMI-based EV Detection & Disaggregation,False,https://www.oracle.com/a/ocom/docs/industries/...
1672,Bidgely19amInsightsRprt,EKLZCYP5,AMI-Driven Insights Report,False,https://www.idcutilitiessummit.com/index/resou...
1674,Hare18disaggHmLdDmdResp,WA8IQAXP,Disaggregation of residential home energy via ...,False,https://dspace.mit.edu/handle/1721.1/117983
1675,Rehman21LoadDisaggThesis,8MAAZN8P,Load Disaggregation: Towards Energy Efficient ...,False,https://openrepository.aut.ac.nz/handle/10292/...


## For raw perplexity dialog markdown

#### "my" version

In [9]:
# # Functions for replacing references in perplexity's dialog copy with links to existing obsidian notes or zotero items 

# def zotero_item_link(zotero_item_key, link_text):
#     """Makes a link to a zotero item, given its key"""
#     return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url):
#     """Convert a URL to a standard form, so that it can be string-compared to the same URL
#     written by a different program, but which is also normalized by this function."""
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

# def replace_perplexity_dialogue_links(perplexity_doc, zot_db_items, output_file):
#     """Replace numeric citations in a perplexity dialog document with links to matching 
#     obsidian literature notes or to zotero items.  A 'match' is determined when the URL 
#     in the perplexity doc matches a zotero item's URL.  Link first to the obsidian literature note
#     when one exists, then try to link to a zotero item.  If neither is available don't change the link.

#     Arguments 
#     perplexity_doc: a full pathlib path to a file of markdown coming from perplexity's copy function
#     zot_db_items: a dataframe with a row of info for every zotero DB item.  The columns are: 
#         citekey: the obsidian note citekey (the stem of its filename)
#         zotkey: zoter item key
#         hasLitNote: true if an obsidian literature note already exists
#         url: the URL associated with this zotero DB item
#     output_file: a full pathlib path to where the output document should go"""

#     # organize the zotero DB info
#     if not isinstance(zot_db_items, pd.DataFrame):
#         raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
#         df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
#         zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

#     zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
#     zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

#     # modify the perplexity dialog doc
#     with open(perplexity_doc, 'r') as mdfile:
#         content = mdfile.read()

#     # Split the content into body and citations
#     parts = content.split("\nCitations:\n")
#     if len(parts) != 2:
#         raise Exception("Couldn't find Citations section")
    
#     body, citations = parts

#     # From citations at doc bottom, get a url for each citation number
#     citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
#     doc_number_to_url = defaultdict(lambda: None, {num:normalize_url(url) for num, url in citation_urls})

#     def make_best_reference_link(doc_url, doc_cite_num):
#         # Replace body citations w/ wikilinks to an obsidian note or if no note, an md link to a zotero item
#         if doc_url and (itemInfo := zot_url_to_item_info[doc_url]) is not None:
#             if itemInfo.hasLitNote:
#                 return f'[[{itemInfo.citekey}]]' # wikilink to obsidian lit note

#             # md link to item in zotero DB
#             link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'  #"bob \u2794 jim"
#             return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
            
#         return f'[{doc_cite_num}]' # not in zotero DB so leave unchanged

#     def replace_body_reference(match):
#         # Replace citations in the body text
#         doc_cite_num = match.group(1)
#         doc_url = doc_number_to_url[doc_cite_num]

#         return ' ' + make_best_reference_link(doc_url, doc_cite_num)

#     body = re.sub(r'\[(\d+)\]', replace_body_reference, body)

#     def replace_citations_reference(match):
#         # Replace citations in the Citations section
#         doc_cite_num = match.group(1)
#         url = match.group(2)
#         doc_url = normalize_url(url)

#         if zot_url_to_item_info[doc_url] is None:
#             return f'[{doc_cite_num}] {doc_url}' # not in zotero DB
#         else:
#             return f'[{doc_cite_num}] =={make_best_reference_link(doc_url, doc_cite_num)}== {url}'

#     citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citations_reference, citations)

#     with open(output_file, 'w') as outfile:
#         outfile.write(body + "\nCitations:\n" + citations)


#### R1-inspired version
(for raw perplexity markdown)

In [10]:
from pathlib import Path
import re
from urllib.parse import urlparse, urlunparse
from collections import defaultdict
import pandas as pd

def zotero_item_link(zotero_item_key: str, link_text: str) -> str:
    """Create a Markdown link to a Zotero item using its library key.
    
    Args:
        zotero_item_key: Zotero item key from library
        link_text: Display text for the link
        
    Returns:
        Markdown link string in format [text](zotero://...)
    """
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def normalize_url(url: str) -> str:
    """Standardize URL format for consistent comparisons.
    
    Converts to lowercase and strips trailing slashes from path.
    
    Args:
        url: Any URL string
        
    Returns:
        Normalized URL string
    """
    parsed = urlparse(url.lower())
    cleaned_path = parsed.path.rstrip('/')
    return urlunparse(parsed._replace(path=cleaned_path))

def replace_perplexity_dialogue_links(
    perplexity_doc: Path,
    zot_db_items: pd.DataFrame,
    output_file: Path
) -> None:
    """Replace numeric citations with Obsidian/Zotero links while preserving original structure.
    
    Processing logic:
    1. Normalizes all URLs for comparison
    2. Creates mapping from normalized URLs to Zotero item metadata
    3. Replaces body citations with appropriate links
    4. Updates citations section with matching references
    
    Args:
        perplexity_doc: Path to input Markdown file
        zot_db_items: DataFrame containing:
            - citekey: Obsidian note ID
            - zotkey: Zotero item key
            - hasLitNote: Boolean for existing literature note 
            - url: Item URL
        output_file: Path for output file
        
    Raises:
        ValueError: If input document structure is invalid
    """
    # Validate input dataframe structure
    required_columns = {'citekey', 'zotkey', 'hasLitNote', 'url'}
    if not required_columns.issubset(zot_db_items.columns):
        missing = required_columns - set(zot_db_items.columns)
        raise ValueError(f"Missing required columns in zot_db_items: {', '.join(missing)}")

    # Preprocess Zotero data
    zot_db_items = zot_db_items.copy()
    zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
    url_to_zot_info = {
        row.url: row 
        for row in zot_db_items.itertuples(index=False)
    }

    # Read and split document
    content = perplexity_doc.read_text(encoding='utf-8')
    try:
        body, citations = content.split("\nCitations:\n", 1)
    except ValueError as e:
        raise ValueError("Invalid document structure - missing citations section") from e

    # Extract citation URLs
    citation_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)
    citation_map = {
        m.group('num'): normalize_url(m.group('url')) 
        for m in citation_matches
    }

    def create_reference(doc_url: str, cite_num: str) -> str:
        """Generate appropriate reference link based on available metadata."""
        if not doc_url:
            return f'[{cite_num}]'
            
        item = url_to_zot_info.get(doc_url)
        if not item:
            return f'[{cite_num}]'

        if item.hasLitNote:
            return f'[[{item.citekey}]]'
            
        link_text = f"{item.citekey}→{item.zotkey[:6]}"
        return zotero_item_link(item.zotkey, link_text)

    # Process body content
    body_processed = re.sub(
        r'\[(\d+)\]',
        lambda m: f' {create_reference(citation_map.get(m.group(1)), m.group(1))}',
        body
    )

    # Process citations section
    def update_citation(m: re.Match) -> str:
        num = m.group('num')
        url = normalize_url(m.group('url'))
        if url in url_to_zot_info:
            # a hack to redo this here
            ref = create_reference(url, num)
            return f'[{num}] =={ref}== {m.group('url')}'
        
        return f'[{num}] {m.group('url')}'

    citations_processed = re.sub(
        r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)',
        update_citation,
        citations,
        flags=re.M
    )

    # Write output
    output_file.write_text(
        f"{body_processed}\nCitations:\n{citations_processed}", 
        encoding='utf-8'
    )

In [11]:
replace_perplexity_dialogue_links(perplexity_dialog_file, zot_db_items, output_file_perplex)
#rfw.ORIG_replace_perplexity_citations_from_perplexity(perplexity_dialog_file, url_to_citekey, output_file_perplex)
ic(perplexity_dialog_file, output_file_perplex)
print('Done.')

ic| perplexity_dialog_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_example.md')
    output_file_perplex: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')


Done.


## For markdown from the "Save my Chatbot" chrome/firefox extension

In [12]:
# This works but only when there is a zotero entry with a matching URL. Not all zotero entries have URLs
#
# import re
# import pandas as pd
# from collections import defaultdict
# from urllib.parse import urlparse, urlunparse

# def zotero_item_link(zotero_item_key, link_text):
#     return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url):
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

# def replace_savemychatbot_links(savemychatbot_doc, zot_db_items, output_file):
#     if not isinstance(zot_db_items, pd.DataFrame):
#         raise Exception('Expected a dataframe. Reading url_to_citekey from file does not yet handle new dataframe column')
    
#     zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
#     zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

#     with open(savemychatbot_doc, 'r') as mdfile:
#         content = mdfile.read()

#     # Split the content into sections
#     sections = re.split(r'(\n---\s*\n\s*\*\*Sources:\*\*\s*\n)', content, flags=re.IGNORECASE)

#     def make_best_reference_link(doc_url, original_text):
#         if doc_url and (itemInfo := zot_url_to_item_info[doc_url]) is not None:
#             if itemInfo.hasLitNote:
#                 return f'[[{itemInfo.citekey}]]'
#             link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'
#             return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
#         return original_text

#     def replace_body_reference(match):
#         full_match = match.group(0)
#         url = normalize_url(match.group(2))
#         return make_best_reference_link(url, full_match)

#     def replace_sources_reference(match):
#         title = match.group(1)
#         url = normalize_url(match.group(2))
#         if zot_url_to_item_info[url] is None:
#             return f'- [{title}]({url})'
#         else:
#             return f'- [{title}]({url}) =={make_best_reference_link(url, title)}=='

#     processed_sections = []
#     for i in range(0, len(sections), 2):
#         body = sections[i]
#         body = re.sub(r'\[(.*?)\]\((https?://\S+)\)', replace_body_reference, body)
#         processed_sections.append(body)

#         if i + 1 < len(sections):
#             sources_header = sections[i + 1]
#             sources = sections[i + 2] if i + 2 < len(sections) else ""
#             sources = re.sub(r'- \[(.*?)\]\((https?://\S+)\)', replace_sources_reference, sources)
#             processed_sections.append(sources_header + sources)

#     with open(output_file, 'w') as outfile:
#         outfile.write(''.join(processed_sections))


In [13]:
# need to combine functions, and didn't completely work, although it almost did.

# import re
# import pandas as pd
# from collections import defaultdict
# from urllib.parse import urlparse, urlunparse
# from typing import List, Dict, Any, Tuple, Optional

# def zotero_item_link(zotero_item_key, link_text):
#     return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url):
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

# def best_zotero_title_match(target_title: str, zotero_items: List[Dict[str, Any]]) -> Tuple[Optional[dict], int]:
#     """Find the title in a list of zotero items that best matches a target title, 
#     return (item, score)."""

#     best_score = 0
#     best_match = None

#     for item in zotero_items:
#         # it's not a zotero item anymore
#         # item_title = item['data'].get('title', '')
#         # score = match_titles(target_title, item_title)

#         score = match_titles(target_title, item['title'])

#         if score > best_score:
#             best_score = score
#             best_match = item

#     return best_match, best_score

# def match_titles(title1: str, title2: str) -> int:
#     # Implement your title matching logic here
#     # This is a placeholder implementation
#     return len(set(title1.lower().split()) & set(title2.lower().split()))

# def find_zotero_item_by_title(title: str, zot_db_items: pd.DataFrame) -> Optional[pd.Series]:
#     """Find the Zotero item with the best matching title."""
    
#     # Convert DataFrame to list of dictionaries
#     zotero_items = zot_db_items.to_dict('records')
    
#     # Use the best_zotero_title_match function
#     best_match, best_score = best_zotero_title_match(title, zotero_items)
    
#     # Check if a match was successfully found (score > 70)
#     if best_score > 70:
#         # Convert the matching item back to a pandas Series
#         return pd.Series(best_match)
#     else:
#         return None

# def find_title_from_sources(url: str, sources_content: str) -> Optional[str]:
#     """Find the title for a given URL in the sources section."""
#     matches = re.findall(r'- \[(.*?)\]\((' + re.escape(url) + r')\)', sources_content)
#     if matches:
#         return re.sub(r'^\(\d+\)\s*', '', matches[0][0]).strip()
#     return None

# def replace_savemychatbot_links(savemychatbot_doc, zot_db_items, output_file):
#     if not isinstance(zot_db_items, pd.DataFrame):
#         raise Exception('Expected a dataframe. Reading url_to_citekey from file does not yet handle new dataframe column')
    
#     zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
#     zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

#     with open(savemychatbot_doc, 'r') as mdfile:
#         content = mdfile.read()

#     # Split the content into sections
#     sections = re.split(r'(\n---\s*\n\s*\*\*Sources:\*\*\s*\n)', content, flags=re.IGNORECASE)

#     def make_best_reference_link(doc_url, original_text, sources_content):
#         if doc_url:
#             itemInfo = zot_url_to_item_info[doc_url]
#             if itemInfo is None:
#                 title = find_title_from_sources(doc_url, sources_content)
#                 if title:
#                     itemInfo = find_zotero_item_by_title(title, zot_db_items)
            
#             if itemInfo is not None:
#                 if itemInfo.hasLitNote:
#                     return f'[[{itemInfo.citekey}]]'
#                 link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'
#                 return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
#         return original_text

#     def replace_body_reference(match, sources_content):
#         full_match = match.group(0)
#         url = normalize_url(match.group(2))
#         return make_best_reference_link(url, full_match, sources_content)

#     def replace_sources_reference(match):
#         title = match.group(1)
#         url = normalize_url(match.group(2))
        
#         # Extract the title without the reference number
#         title_without_number = re.sub(r'^\(\d+\)\s*', '', title).strip()
        
#         if zot_url_to_item_info[url] is not None:
#             itemInfo = zot_url_to_item_info[url]
#         else:
#             itemInfo = find_zotero_item_by_title(title_without_number, zot_db_items)
        
#         if itemInfo is not None:
#             link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'
#             zotero_link = zotero_item_link(itemInfo.zotkey, link_text)
#             return f'- [{title}]({url}) =={zotero_link}=='
#         else:
#             return f'- [{title}]({url})'

#     processed_sections = []
#     for i in range(0, len(sections), 2):
#         body = sections[i]
#         sources_content = sections[i + 2] if i + 2 < len(sections) else ""
#         body = re.sub(r'\[(.*?)\]\((https?://\S+)\)', lambda m: replace_body_reference(m, sources_content), body)
#         processed_sections.append(body)

#         if i + 1 < len(sections):
#             sources_header = sections[i + 1]
#             sources = sources_content
#             sources = re.sub(r'- \[(.*?)\]\((https?://\S+)\)', replace_sources_reference, sources)
#             processed_sections.append(sources_header + sources)

#     with open(output_file, 'w') as outfile:
#         outfile.write(''.join(processed_sections))

# # # Example usage
# # savemychatbot_doc = 'path/to/your/savemychatbot_doc.md'
# # zot_db_items = pd.read_csv('path/to/your/zotero_database.csv')  # Assuming the Zotero database is in CSV format
# # output_file = 'path/to/your/output_file.md'

# # replace_savemychatbot_links(savemychatbot_doc, zot_db_items, output_file)


In [22]:
import re
import pandas as pd
from typing import Optional, Dict, List, Tuple
from collections import Counter

# Placeholder for the rfw.match_titles function
# def match_titles(title1: str, title2: str) -> int:
#     """Simulate rfw.match_titles function to return a similarity score."""
#     return len(set(title1.lower().split()) & set(title2.lower().split())) * 100 // max(len(title1.split()), len(title2.split()))

def zotero_item_link(zotero_item_key, link_text):
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url: str) -> str:
#     """Normalize a URL by ensuring consistent formatting."""
#     from urllib.parse import urlparse, urlunparse
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

def find_zotero_item_by_url(url: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find a Zotero item by its URL."""
    normalized_url = rfw.normalize_url(url)
    matches = zot_db_items[zot_db_items['url'] == normalized_url]
    if not matches.empty:
        return matches.iloc[0].to_dict()  # Return as a dictionary
    return None

def find_zotero_item_by_title(title: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find the Zotero item with the best matching title."""
    zotero_items = zot_db_items.to_dict('records')
    best_match = None
    best_score = 0

    for item in zotero_items:
        score = rfw.match_titles(title, item['title'], main_title_only=False)
        if score > best_score:
            best_match = item
            best_score = score

    if best_score > 70:  # Threshold for a good match
        ic(best_score, title, item['title'])
        return best_match  # Return as a dictionary
    
    return None

def build_source_url_to_title(sources_content: str) -> Dict[str, str]:
    """Build a dictionary mapping URLs to titles from the sources section."""
    source_url_to_title = {}
    matches = re.findall(r'- \[(.*?)\]\((https?://\S+)\)', sources_content)
    for title, url in matches:
        normalized_url = rfw.normalize_url(url)
        title = re.sub(r'^\s*\(\d+\)\s*', '', title) # remove ref num
        source_url_to_title[normalized_url] = title.strip()
    return source_url_to_title

def replace_links_with_zotero_items(
    body_content: str,
    sources_content: str,
    zot_db_items: pd.DataFrame,
) -> Tuple[str, str, Counter]:
    """
    Replace links in body content and sources content with Zotero links or leave them as-is.
    
    Returns:
        - Updated body content.
        - Updated sources content.
        - A Counter of URLs in the body that were not found in the sources.
    """
    
    # Build source URL-to-title mapping
    source_url_to_title = build_source_url_to_title(sources_content)

    missing_sources_links = Counter()

    def best_link_from_zotero_item(zotero_item):
        if zotero_item.get('hasLitNote', False):
            return f'[[{zotero_item["citekey"]}]]'
        else:
            # TODO: error check for missing citekey or zotkey
            link_text = f'{zotero_item["citekey"]}→{zotero_item["zotkey"]}'
            return zotero_item_link(zotero_item["zotkey"], link_text)

    def best_replacement_link_by_match(url):

        if zotero_item := find_zotero_item_by_url(url, zot_db_items):
            return best_link_from_zotero_item(zotero_item)

        if url in source_url_to_title:
            # try to replace matching source link title with zotero item title
            title = source_url_to_title[url]
            if zotero_item := find_zotero_item_by_title(title, zot_db_items):
                return best_link_from_zotero_item(zotero_item)

        return None # no kind of zotero item match
            
    def replace_body_link(match):
        url = normalize_url(match.group(2))
        if url not in source_url_to_title:
            missing_sources_links[url] += 1 # for later error reporting

        if match_link := best_replacement_link_by_match(url):
            return match_link
        
        return match.group(0)  # No Zotero item is found, leave the link as-is

    def replace_sources_link(match):
        title = match.group(1)
        url = normalize_url(match.group(2))

        if match_link := best_replacement_link_by_match(url):
            return f'- [{title}]({url}) =={match_link}=='

        return match.group(0)

    # Replace links in the body content
    updated_body_content = re.sub(r'\[(.*?)\]\((https?://\S+)\)', replace_body_link, body_content)

    # Replace links in the sources content
    updated_sources_content = re.sub(r'- \[(.*?)\]\((https?://\S+)\)', replace_sources_link, sources_content)

    return updated_body_content, updated_sources_content, missing_sources_links

def process_markdown_file(input_file: str, output_file: str, zot_db_items: pd.DataFrame):
    """Process a markdown file to replace links with Zotero references."""
    
    with open(input_file, 'r') as infile:
        content = infile.read()

    # Split content into sections based on level 2 headers named "User"
    sections = re.split(r'(?<=\n)## User', content)

    processed_sections = []
    log_missing_links = []

    for section_idx, section in enumerate(sections[1:], start=1):  # Skip anything before the first "User" section
        # Split each section into body and sources parts
        parts = re.split(r'(\n---\s*\n\s*\*\*Sources:\*\*\s*\n)', section)
        
        if len(parts) < 3:
            continue  # Skip sections without both body and sources

        body_content = parts[0]
        #sources_header_and_content = parts[1] + parts[2]
        
        # Process body and sources content to replace links with Zotero references
        updated_body_content, updated_sources_content, missing_sources_links_counter = replace_links_with_zotero_items(
            body_content,
            parts[2],
            zot_db_items,
        )

        # Log missing links for this section (with counts)
        if missing_sources_links_counter:
            log_missing_links.append(
                f"Section {section_idx}: Missing links - " +
                ", ".join([f"{url} (count: {count})" for url, count in missing_sources_links_counter.items()])
            )

        processed_sections.append(f"## User{updated_body_content}")
        processed_sections.append(parts[1])  # Sources header remains unchanged
        processed_sections.append(updated_sources_content)

    # Write processed content to output file
    with open(output_file, 'w') as outfile:
        outfile.write(''.join(processed_sections))

    # Print log of missing links at the end
    if log_missing_links:
        print("Log of missing links:")
        for log_entry in log_missing_links:
            print(log_entry)

""" # Example usage
if __name__ == "__main__":
    input_markdown_file = "example.md"
    output_markdown_file = "output.md"

    # Example Zotero database (as a Pandas DataFrame)
    zot_db_items_df = pd.DataFrame([
        {'url': 'https://example.com', 'title': 'Example Title', 'citekey': 'ExampleCiteKey', 'zotkey': 'Z12345', 'hasLitNote': True},
        {'url': '', 'title': 'Another Example Title', 'citekey': 'AnotherCiteKey', 'zotkey': '
 """

' # Example usage\nif __name__ == "__main__":\n    input_markdown_file = "example.md"\n    output_markdown_file = "output.md"\n\n    # Example Zotero database (as a Pandas DataFrame)\n    zot_db_items_df = pd.DataFrame([\n        {\'url\': \'https://example.com\', \'title\': \'Example Title\', \'citekey\': \'ExampleCiteKey\', \'zotkey\': \'Z12345\', \'hasLitNote\': True},\n        {\'url\': \'\', \'title\': \'Another Example Title\', \'citekey\': \'AnotherCiteKey\', \'zotkey\': \'\n '

In [23]:

ic(savemychatbot_perplexity_dialog_file, output_file_savemychatbot)
process_markdown_file(savemychatbot_perplexity_dialog_file, output_file_savemychatbot, zot_db_items)
print('Done.')

ic| savemychatbot_perplexity_dialog_file: 

WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_multi_prompt_savemychatbot_example.md')
    output_file_savemychatbot: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_savemychatbot_multiprompt_perplexity_example.md')
ic| normalized_url: 'https://en.wikipedia.org/wiki/mutual_information'
    title: 'Mutual information'
ic| normalized_url: 'https://people.cs.umass.edu/~elm/teaching/docs/mutinf.pdf'
    title: 'PDF Entropy and Mutual Information'
ic| normalized_url: 'https://www.blog.trainindata.com/mutual-information-with-python'
    title: "Mutual information with Python - Train in Data's Blog"
ic| normalized_url: 'https://quantdare.com/what-is-mutual-information'
    title: 'What is Mutual Information? - Quantdare'
ic| normalized_url: 'http://www.scholarpedia.org/article/mutual_information'
    title: 'Mutual information - Scholarpedia'
ic| normalized_url: 'https://www.nature.com/articles/srep10981'
    title: 'Mutual Information bet

KeyboardInterrupt: 